In [20]:
import nest_asyncio
import os
import pickle
from llama_index.core import VectorStoreIndex

In [21]:
with open("embeddings.pkl", "rb") as f:
    embeds = pickle.load(f)

In [22]:
from pinecone.grpc import PineconeGRPC
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext

In [23]:
from dotenv import load_dotenv
from pinecone import ServerlessSpec
load_dotenv()
api = os.getenv("PINECONE_API_KEY")

pine = PineconeGRPC(api_key=api)


In [24]:
pine.create_index("embeddings2", dimension=1024, metric='dotproduct', spec=ServerlessSpec(cloud="aws",region="us-east-1"))

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'application/json', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '61134451bbf56dd88fa66bbaf8b91d56', 'date': 'Thu, 01 Oct 2026 12:35:26 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [25]:
with open("extracts.pkl", "rb") as f:
    nodes = pickle.load(f)

In [26]:
index = pine.Index("embeddings2")

In [27]:
from llama_index.sparse_embeddings.fastembed import FastEmbedSparseEmbedding

sparse_model = FastEmbedSparseEmbedding(model_name="prithivida/Splade_PP_en_v1")

In [28]:
vectorstore = PineconeVectorStore(pinecone_index=index, add_sparse_vector=True, sparse_embedding_model=sparse_model)

In [29]:
vectors=[]
for i, embed in enumerate(embeds):
    nodal = {
        "id" : f"{i}",
        "values": embed,
        "metadata": { "text": nodes[i].text, **nodes[i].metadata}
        
    }
    vectors.append(nodal)


In [30]:
for i in range(0,len(vectors), 100):
    batch = vectors[i:i+100]
    if i == 9500:
        batch = vectors[i:]
    index.upsert(vectors=batch)
    print("upserted ",i)

upserted  0
upserted  100
upserted  200
upserted  300
upserted  400
upserted  500
upserted  600
upserted  700
upserted  800
upserted  900
upserted  1000
upserted  1100
upserted  1200
upserted  1300
upserted  1400
upserted  1500
upserted  1600
upserted  1700
upserted  1800
upserted  1900
upserted  2000
upserted  2100
upserted  2200
upserted  2300
upserted  2400
upserted  2500
upserted  2600
upserted  2700
upserted  2800
upserted  2900
upserted  3000
upserted  3100
upserted  3200
upserted  3300
upserted  3400
upserted  3500
upserted  3600
upserted  3700
upserted  3800
upserted  3900
upserted  4000
upserted  4100
upserted  4200
upserted  4300
upserted  4400
upserted  4500
upserted  4600
upserted  4700
upserted  4800
upserted  4900
upserted  5000
upserted  5100
upserted  5200
upserted  5300
upserted  5400
upserted  5500
upserted  5600
upserted  5700
upserted  5800
upserted  5900
upserted  6000
upserted  6100
upserted  6200
upserted  6300
upserted  6400
upserted  6500
upserted  6600
upserte

In [31]:
from llama_index.llms.groq import Groq
load_dotenv()

True

In [32]:
api = os.getenv("GROQ_API_KEY")
llm = Groq(model="openai/gpt-oss-120b", api=api)

In [33]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer

In [34]:
from llama_index.core import Settings
Settings.llm = llm

from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
import nest_asyncio

nest_asyncio.apply()
embed_model = HuggingFaceInferenceAPIEmbedding(
    model_name="BAAI/bge-m3",
    token=os.getenv("HF_API_KEY"),
)
Settings.embed_model=embed_model

vectorindex = VectorStoreIndex.from_vector_store(vector_store=vectorstore, llm=llm)



In [39]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

filter = MetadataFilters(filters=[ExactMatchFilter(key="classification", value="paper_body")])

In [40]:


retriver = VectorIndexRetriever(index=vectorindex,verbose=True, similarity_top_k=20, vector_store_query_mode="hybrid", filters=filter)
synth = get_response_synthesizer()



In [41]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L4-v2", top_n=7)

query = input("Enter your query: ")

prompt_instructions = f'You are a part of a conservation research RAG assistant - improve the prompt for the LLM to yield better results. The user has entered the prompt below:\n {query} \n The prompt could have limitations such as being too specific where the corpus may not have information on the exact species or too generalised for a corpus detailing technical insights into species conservation. It may also not provide constraints and rules to the LLM \n Your job is to enhance the query so that RAG retrieval returns interesting points of discussion. Ensure you mention that all information must be backed by citations sourced only from the actual RAG vector database, tell it to recognise levels of uncertainty of information due to the corpus and a breakdown of information reasoning which does not go beyond species conservation. \n in Ensure the new prompt is between 50-60 words max. \nOnly return the prompt text.'

new_prompt = llm.complete(prompt_instructions).text
print(new_prompt)

query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth, node_postprocessors=[rerank])
response = query_engine.query(new_prompt)
nodes = response.source_nodes

print([node.score for node in nodes])

Provide a concise overview of white‑tiger conservation, including current threats, population trends, and effective management strategies. Cite each fact using only sources retrieved from the RAG vector database, indicate any uncertainty levels, and present a brief reasoning breakdown. Discuss genetic health, habitat connectivity, and community involvement. Limit discussion strictly to species‑conservation aspects.
[-2.6586363, -4.170266, -4.171733, -4.5025787, -4.5653343, -4.790634, -5.037864]


In [42]:
print([node.text for node in nodes])

['Objectives   \n48 \nenforcement, allowing over-exploitation (a threat) to continue. \n \nLike many other aspects of strategic planning for species conservation, problem analyses \nare likely to be most useful when conducted by interdisciplinary teams of stakeholders with \na diverse array of expertise and experience.  This should ensure that all of the key threats \nand constraints are incorporated into the analysis.  In a workshop setting, threats will \nusually have been identified as part of the Status Review, while the constraints may be \nidentified through sharing of the group’s collective knowledge of the social, economic, \ninformation, and policy environment within which conservation efforts would be conducted.  \nSince the problem analysis seeks to identify such a broad array of problems, wherever \npossible all workshop participants should be involved in conducting the analysis.  In many \ncases, species specialists will be most aware of the proximate threats to the specie

In [43]:
print(response)

The provided excerpts do not contain any information on white‑tiger (Panthera tigris tigris) conservation—there are no details about current threats, population trends, genetic health, habitat connectivity, or community involvement. Consequently, I cannot supply a factual overview of white‑tiger conservation based on the available sources.

**Uncertainty level:** High (the required information is absent from the supplied material).


In [1]:
query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth)
query = input("Enter your query: ")
response = query_engine.query(new_prompt)
print(response)

NameError: name 'RetrieverQueryEngine' is not defined

In [ ]:

"""
schema for design: 

user enters: conservation techniques for this and that

rephrase with prompt: (query expansion)
provide a sophisticated analysis of the point of disucssion. Follow the schema provided to generate a thoughtful report of the question based on the corpus. Ensure that all claims are referenced with page number and file name. Draw conclusions based on the information from the corpus. if you cannot find any reasonable link with the corpus, make it clear that there are gaps in place. 

schema: 
{
    'Introduction' : pass
    'Key insights and strategies':
    'Confidence score /10':
    'List of statistics':
    'References':
}

new prompt is taken and entered into the vector db. Results returned. 

Query stratgies - reranking, 

"""